# 4 · Precomputed candidates · the `aud:` STRING

SINTER tightening narrowed Redis-side work to one round trip, but the
ranker still receives ~50 candidates and the eligibility filter still has
to run online for every one of them.

The next move: do the candidate generation **offline**, once per MAID, and
store the result. At bid time the engine just `GET`s a precomputed list
and skips the SINTER step entirely.

This is the central pattern of the prototype.


In [1]:
# Locate the repo root so `notebooks._demo_setup` is importable regardless
# of where the kernel was launched (the package layout requires the repo
# root on sys.path).
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## What's in `aud:{maid_id}`

A single Redis STRING per MAID, containing a JSON list of campaign IDs
that survive the user's *static* targeting. "Static" here means everything
that doesn't depend on live state: geo, device, card_tier, segment
required/any_of/none_of.


In [2]:
import json
raw = client.get('aud:maid_00042')
candidates = json.loads(raw)
print(f'aud:maid_00042 has {len(candidates)} candidates')
print(f'first 10: {candidates[:10]}')
print(f'string size on the wire: {len(raw)} bytes')

aud:maid_00042 has 27 candidates
first 10: ['c01365', 'c01363', 'c01011', 'c00722', 'c00848', 'c01927', 'c02169', 'c00099', 'c02229', 'c00763']
string size on the wire: 270 bytes


## How the precompute is built

`data.hybrid_precompute._matches_static_targeting` is the function the
batch job runs against every (MAID, campaign) pair. The function is
deliberately a subset of the online filter — it leaves out everything
that can change between batch runs (pacing, budget, frequency cap,
taxonomy_filter against possibly-stale interest scores).


In [3]:
import inspect
from data import hybrid_precompute

print(inspect.getsource(hybrid_precompute._matches_static_targeting))

def _matches_static_targeting(user: UserProfile, campaign: Campaign) -> bool:
    """Match the static targeting fields used by the precompute.

    Note: the per-campaign `taxonomy_filter` is *not* evaluated here. Taxonomy
    scores can drift between batch precompute and bid time (an online
    feedback path may rewrite individual labels between batches), so the
    filter must be evaluated online against the current `maid_hot:{maid_id}`
    interests. The per-MAID candidate list therefore over-approximates by
    the taxonomy_filter pass rate; the online taxonomy mode
    (`hybrid_bitmap_taxonomy`) closes that gap.
    """
    user_segments = set(user.segments)
    return (
        _matches_dimension(user.geo, campaign.geo)
        and _matches_optional_list(user.state, campaign.geo_states)
        and _matches_optional_list(user.postal_code, campaign.geo_postal_codes)
        and _matches_dimension(user.device, campaign.device)
        and _matches_dimension(user.device_type, campai

The precompute is regenerated by `data.synthetic.generate_dataset` every
time the synthetic data is rebuilt. In production, the same idea applies
at a different cadence — DNA pipeline rebuilds the precompute periodically
and writes one `aud:{maid_id}` per MAID into Redis with a versioned
prefix swap.


## The precomputed_segment bid path

With the precompute in place, the bid path collapses to:

1. resolve identity,
2. fetch the hot scoring profile (`maid_hot:`),
3. **`GET aud:{maid_id}`** — one round trip, returns ~30 candidate IDs,
4. pipelined `HGETALL campaign:{id}` for each candidate,
5. apply minimal live gating (pacing, budget, frequency),
6. rerank.

Compare to the `full_realtime` path in notebook 2 — same steps, but
candidate generation is now a `GET` instead of a SCAN+HGETALL of all
2500 campaigns.


In [4]:
from app.models import ScoringProfile, Campaign
from app.candidate import filter_campaigns_for_user
from app.ranking import rerank_campaigns

IDENTITY_TOKEN = 'id_00042_01'
timer = StepTimer()

with timer.step('identity_resolution'):
    maid_id = client.get(f'identity:{IDENTITY_TOKEN}')

with timer.step('hot_profile_fetch'):
    payload = client.hmget(f'maid_hot:{maid_id}', 'user_id', 'interests_json', 'impression_count')
    scoring = ScoringProfile.from_redis_hash({
        'user_id': payload[0],
        'interests_json': payload[1],
        'impression_count': payload[2],
    })

with timer.step('aud_get'):
    raw = client.get(f'aud:{maid_id}')
    candidate_ids = json.loads(raw)

with timer.step('campaign_fetch_pipelined'):
    pipe = client.pipeline(transaction=False)
    for cid in candidate_ids:
        pipe.hgetall(f'campaign:{cid}')
    campaigns = [Campaign.from_redis_hash(p) for p in pipe.execute() if p]

with timer.step('fcap_fetch'):
    fcap_counts_raw = client.hmget(f'fcap:{maid_id}', candidate_ids) if candidate_ids else []
    fcap_counts = {
        cid: int(v) for cid, v in zip(candidate_ids, fcap_counts_raw or [])
        if v is not None
    }

with timer.step('minimal_live_filter'):
    eligible = [
        c for c in campaigns
        if c.pacing_status == 'active'
        and c.spent_today_usd < c.daily_budget_usd
        and fcap_counts.get(c.campaign_id, 0) < c.frequency_cap
    ]

with timer.step('rerank'):
    top_5 = rerank_campaigns(scoring, eligible, top_k=5)

print(f'maid_id        = {maid_id}')
print(f'candidates     = {len(candidate_ids)}  (vs 2500 in full_realtime)')
print(f'eligible       = {len(eligible)}')
print(f'top 5: {[(r.campaign_id, round(r.score, 4)) for r in top_5]}')
print()
print(timer.summary())

maid_id        = maid_00042
candidates     = 27  (vs 2500 in full_realtime)
eligible       = 17
top 5: [('c01011', 5.8237), ('c00848', 4.8107), ('c01551', 4.4065), ('c01222', 4.2812), ('c02229', 4.1223)]

             identity_resolution    0.529 ms
               hot_profile_fetch    0.326 ms
                         aud_get    0.312 ms
        campaign_fetch_pipelined    1.029 ms
                      fcap_fetch    1.135 ms
             minimal_live_filter    0.006 ms
                          rerank    0.079 ms
--------------------------------------------
                           TOTAL    3.416 ms


## How this compares to the SINTER paths

| mode | candidate generation | round trips | candidates returned |
| --- | --- | ---: | ---: |
| `full_realtime` | SCAN + HGETALL all | 1 (pipelined) | 2500 |
| `maid_bruteforce_sinter` | 26 SINTER probes (sequential) | ~28 | ~50 |
| `maid_tightened_sinter` | 3 SINTER probes (pipelined) | ~3 | ~50 |
| `precomputed_segment` | `GET aud:{maid_id}` | ~3 (incl. `maid_hot` + fcap) | ~30 |

The precompute path is doing **less Redis work and less app work** than
either SINTER mode, because the offline batch did the SET algebra ahead of
time. On a tuned VM the precomputed_segment decision-path lands at
`~2.7 ms` p50 — about half the tightened SINTER path.


## One caveat about the precompute

The precompute deliberately leaves out anything that can change between
batch runs:

- **pacing / budget**: campaigns can run out mid-day; the bid path has to
  re-check `campaign_state:` (or use the bitmap gate from notebook 5).
- **frequency cap**: per-MAID-per-campaign counter, written every win;
  the bid path checks `fcap:{maid_id}` online.
- **taxonomy_filter**: per-campaign AND/OR/NOT on float interest scores;
  the bid path evaluates this online from the user's `interests` because
  the scores can drift between batch runs (notebook 6).

`precomputed_segment` does the minimal live gating (pacing + budget + freq).
`hybrid_precompute_plus_realtime` does the *full* live gating, including
exact targeting and the taxonomy filter. The next two notebooks tighten
the live-gating step further.
